# 05 — Índices Espectrais por Classe LCZ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/05_spectral_indices.pt.ipynb)

**Objetivo de aprendizagem.** Ao final deste notebook você terá baixado uma pequena cena Sentinel-2 de uma cidade via Microsoft Planetary Computer, calculado uma bateria de índices espectrais (NDVI, NDBI e afins) a partir de suas bandas, e resumido como esses índices variam entre as classes de Zona Climática Local (LCZ) — com estatísticas rigorosas de tamanho de efeito, não apenas boxplots observados a olho nu.

## Por que índices espectrais importam para análise de LCZ

O esquema de Zonas Climáticas Locais (Stewart & Oke, 2012) classifica paisagens urbanas e naturais em
17 classes padronizadas com base em sua estrutura de superfície (altura de edifícios/árvores,
espaçamento, impermeabilização) e cobertura do solo. É uma classificação *morfológica* poderosa —
mas a morfologia sozinha não diz o que realmente está crescendo, construído ou exposto no solo em
uma determinada zona. Bairros de "LCZ 6 — baixo aberto" de duas cidades podem parecer idênticos em
um modelo 3D, mas ter quantidades muito diferentes de árvores de rua, gramados ou superfície
pavimentada — e essa diferença importa enormemente para como cada zona realmente se comporta
termicamente.

É aqui que os índices espectrais entram. Calculados pixel a pixel a partir da reflectância de
superfície de satélites, eles fornecem uma leitura direta e fisicamente fundamentada da composição
da cobertura do solo:

- **NDVI** (Índice de Vegetação por Diferença Normalizada) — a proxy padrão para fração de
  vegetação verde e vigor do dossel. A vegetação reflete fortemente o infravermelho próximo (NIR) e
  absorve luz vermelha para fotossíntese, então `(NIR - Vermelho) / (NIR + Vermelho)` é alto sobre
  dossel saudável e baixo/negativo sobre água, solo exposto ou superfícies impermeáveis.
- **NDBI** (Índice de Área Construída por Diferença Normalizada) — a proxy complementar para fração
  de superfície impermeável/construída, explorando o fato de que materiais construídos refletem
  mais infravermelho de ondas curtas (SWIR) do que NIR, ao contrário da vegetação.
- Índices de água (NDWI, MNDWI), índices de solo (BSI) e outros — cada um isola um componente
  diferente do mosaico de cobertura do solo que um único código LCZ comprime em uma categoria.

Por que isso importa para a ciência do clima urbano? Porque a fração de vegetação e a fração
impermeável são dois dos *principais fatores físicos* do efeito de Ilha de Calor Urbana que o
esquema LCZ foi projetado para ajudar a explicar (Stewart & Oke, 2012; Anjos, M. et al., 2025,
*Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0). NDVI e NDBI por classe
LCZ permitem ir de "LCZ 2 (midrise compacto) e LCZ 6 (baixo aberto) são estruturalmente diferentes"
para "LCZ 2 tem mensuravelmente menos vegetação e mais superfície construída que LCZ 6, com tamanho
de efeito d = X" — uma afirmação quantitativa, falseável e comparável entre cidades. O d de Cohen,
usado ao longo deste notebook, expressa a diferença entre as médias de dois grupos em unidades de
desvio padrão combinado: por convenção, |d| < 0,2 é negligível, 0,2-0,5 é pequeno, 0,5-0,8 é médio, e
> 0,8 é uma diferença grande e praticamente importante. Essa é uma base muito mais defensável para
recomendações de política ou design ("priorizar quadras LCZ 2 para retrofits de infraestrutura
verde") do que uma comparação visual de boxplots.

Este notebook constrói o pipeline completo: `lcz_get_map` → `lcz_get_planetary_computer` (cena
Sentinel-2) → `lcz_get_indices` (índices espectrais) → `lcz_cal_indices` (estatísticas e gráficos por
LCZ). Também mostra `lcz_cal_indexes`, uma ferramenta de estatística zonal mais geral (coberta
integralmente no notebook 07) que pode resumir *qualquer* variável em grade por classe LCZ —
reaproveitando uma banda NDVI como exemplo rápido de por que você não precisa de uma função dedicada
toda vez que quer um resumo por classe de um raster.

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git"


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## 1. Obter o mapa LCZ

`lcz_get_map(city=...)` geocodifica a cidade, baixa apenas a janela que intersecta o raster LCZ
global (Demuzere et al., 2022) do Zenodo, e armazena em cache tanto o limite da cidade quanto o
GeoTIFF recortado localmente. Usamos **Juiz de Fora** (capital de Liechtenstein) — uma cidade minúscula,
para que o download do Sentinel-2 no próximo passo permaneça rápido.

In [2]:
from LCZ4py import lcz_get_map

lcz_map_path = lcz_get_map(city="Juiz de Fora")
print(lcz_map_path)

06:27:19 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


O caminho retornado é um GeoTIFF com uma banda de códigos de classe LCZ (1-17), recortado ao limite administrativo de Juiz de Fora. Todos os passos seguintes deste notebook usam esse mesmo raster como grade de referência — índices e variáveis em grade são todos reprojetados/recortados sobre ele.

## 2. Baixar uma cena Sentinel-2 via Planetary Computer

`lcz_get_planetary_computer` transmite ativos STAC do Microsoft Planetary Computer recortados à
área do mapa LCZ — sem necessidade de autenticação manual (as coleções públicas do Planetary
Computer não exigem chave). Parâmetros principais:

- `collection="sentinel-2-l2a"` — reflectância de superfície corrigida atmosfericamente,
  resolução de ~10-60 m.
- `assets` — quais bandas buscar. O atalho embutido só pega B04/B03/B02/B08 (vermelho/verde/azul/
  NIR); adicionamos `"B11"` (SWIR1) explicitamente para que `lcz_get_indices` também possa calcular
  NDBI e MNDWI, não apenas índices da família NDVI.
- `start_date`/`end_date` — uma janela curta mantém a busca STAC e o download rápidos.
- `max_cloud_cover` — filtra cenas nubladas.
- `max_items` — quantos itens STAC (cenas) mosaicar/compor. Mantemos isso pequeno (2) para um
  download rápido e adequado ao Colab; aumente para um composto mais completo e livre de nuvens em
  trabalho real.

Esta célula pode ser lenta dependendo da carga do Planetary Computer e das condições de rede — isso
é esperado para um download real de satélite, mesmo que pequeno.

In [3]:
from LCZ4py import lcz_get_planetary_computer

s2 = lcz_get_planetary_computer(
    lcz_map_path,
    collection="sentinel-2-l2a",
    assets=["B04", "B03", "B02", "B08", "B11"],  # +B11 (SWIR1) libera NDBI/MNDWI
    start_date="2023-06-01",
    end_date="2023-08-31",
    max_cloud_cover=30.0,
    max_items=2,
)
print(s2.path)
print(s2.bands)

06:27:19 - LCZ4py.general.lcz_get_planetary_computer - INFO - Using cached Planetary Computer stack: /Users/co2map/.lcz4r_cache/pc_sentinel-2-l2a_79596b847ae2c600.tif


/Users/co2map/.lcz4r_cache/pc_sentinel-2-l2a_79596b847ae2c600.tif
['B04', 'B03', 'B02', 'B08', 'B11']


`s2` é um `LCZPCResult` cujo `.path` aponta para um GeoTIFF multibanda (uma banda por ativo solicitado), já recortado e alinhado pixel a pixel à grade do mapa LCZ, e cujo `.bands` lista os nomes dos ativos na ordem das bandas. Como `cache=True` é o padrão, esse GeoTIFF é persistido em disco — necessário para `lcz_get_indices`, que relê do disco em vez de confiar apenas no array em memória.

## 3. Calcular índices espectrais

`lcz_get_indices` lê a pilha de bandas, mapeia nomes de banda específicos do sensor (`B04`, `B08`,
...) para papéis canônicos (vermelho, nir, swir1, ...) automaticamente, e calcula todo índice cujas
bandas necessárias estejam presentes — ou uma lista restrita `indices=[...]`. Com B04/B03/B02/B08/B11
disponíveis podemos calcular toda a família NDVI mais NDBI, MNDWI e outros (qualquer coisa que
precise de SWIR2 ou uma banda térmica não está disponível aqui — o Sentinel-2 não tem banda térmica,
e não buscamos B12).

Restringimos o pedido a quatro índices amplamente usados para uma análise focada:

- **NDVI** — fração/vigor de vegetação (família vegetação).
- **NDBI** — fração de área construída/impermeável (família urbana).
- **MNDWI** — fração de água aberta (família água, usando SWIR1 — mais robusto contra falsos
  positivos de área construída do que o NDWI simples).
- **EVI** — índice de vegetação aprimorado, corrige parte da sensibilidade do NDVI a atmosfera e
  fundo de solo, útil como checagem cruzada do NDVI em dossel denso.

In [4]:
from LCZ4py import lcz_get_indices

idx = lcz_get_indices(s2, indices=["NDVI", "NDBI", "MNDWI", "EVI"])
print(idx.indices)
print(idx.array.shape)  # (n_indices, H, W)

06:27:19 - LCZ4py.general.lcz_get_indices - INFO - Computed {val NDVI}.


06:27:19 - LCZ4py.general.lcz_get_indices - INFO - Computed {val NDBI}.


06:27:19 - LCZ4py.general.lcz_get_indices - INFO - Computed {val MNDWI}.


06:27:19 - LCZ4py.general.lcz_get_indices - INFO - Computed {val EVI}.


['NDVI', 'NDBI', 'MNDWI', 'EVI']
(4, 120, 131)


`idx.array` é uma pilha float32 `(4, H, W)`, uma banda por índice solicitado, NaN onde a pilha Sentinel-2 de origem era NaN (já limitada à área de Juiz de Fora). `idx.indices` dá a ordem das bandas. Neste ponto cada pixel tem uma pontuação de vegetação, uma de área construída e uma de água — mas ainda não relacionadas à classe LCZ. Esse é o trabalho de `lcz_cal_indices`.

## 4. Estatísticas por classe LCZ: boxplots

`lcz_cal_indices` é a contraparte estatística/de visualização de `lcz_get_indices`: agrupa cada
pixel válido (classes LCZ 1-17) por classe e calcula estatísticas descritivas (contagem, média,
mediana, desvio padrão, mín/máx, quartis, e um intervalo de confiança da média) por par (índice,
classe LCZ). Com `plot_type="box"` renderiza um boxplot Plotly de estatísticas pré-computadas por
índice — um subplot por índice quando vários são solicitados (small-multiples), colorido com a
paleta oficial LCZ.

In [5]:
from LCZ4py import lcz_cal_indices

stats_box = lcz_cal_indices(lcz_map_path, idx, plot_type="box")
stats_box.df.head(10)

lcz,count,mean,median,std,min,max,q25,q75,parameter,ci_lower,ci_upper,lcz_name,lcz_col,lcz_colorblind,type
i16,u64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,str,str,str,str
5,101,0.262641,0.259452,0.093903,0.079074,0.484422,0.210948,0.32899,"""NDVI""",0.244103,0.281178,"""Open midrise""","""#FF6628""","""#989600""","""Built-up"""
6,306,0.330289,0.338026,0.106936,-0.043363,0.569071,0.27538,0.39809,"""NDVI""",0.318259,0.342318,"""Open lowrise""","""#FF985E""","""#739F00""","""Built-up"""
8,22,0.221593,0.196923,0.128487,0.080021,0.5624,0.10127,0.334136,"""NDVI""",0.164625,0.278561,"""Large lowrise""","""#BBBBBB""","""#00AA63""","""Built-up"""
9,119,0.382647,0.421838,0.153795,-0.066413,0.635571,0.322666,0.489563,"""NDVI""",0.354728,0.410565,"""Sparsely built""","""#FFCBAB""","""#00AD89""","""Built-up"""
11,1260,0.519146,0.530247,0.103436,0.035975,0.722769,0.470538,0.596331,"""NDVI""",0.513429,0.524863,"""Dense trees""","""#006A18""","""#00A7C5""","""Natural"""
12,88,0.538883,0.555472,0.106647,0.216568,0.69024,0.481661,0.631097,"""NDVI""",0.516287,0.561479,"""Scattered trees""","""#00A926""","""#009EDA""","""Natural"""
14,244,0.35646,0.372455,0.153747,-0.070995,0.671183,0.260501,0.471427,"""NDVI""",0.337072,0.375848,"""Low plants""","""#B5DA7F""","""#9E7FE5""","""Natural"""
15,389,0.437761,0.504288,0.18151,0.065242,0.692406,0.257456,0.580176,"""NDVI""",0.419667,0.455854,"""Bare rock or paved""","""#000000""","""#C36FDA""","""Natural"""
17,10,0.007838,-0.014479,0.064743,-0.057543,0.167604,-0.027836,0.039481,"""NDVI""",-0.038476,0.054152,"""Water""","""#656BFA""","""#E264A9""","""Natural"""


**Como interpretar:** cada caixa é a distribuição de pixels de uma classe LCZ para um índice. Para NDVI, espere que classes naturais (11-17, especialmente LCZ D — plantas baixas, e LCZ A — árvores densas) fiquem bem acima das classes urbanas (1-10), com LCZ 1-3 (compacto alto/médio/baixo) mostrando as caixas de NDVI mais baixas e estreitas. Para NDBI o padrão tipicamente se inverte: classes urbanas compactas mostram NDBI mais alto (frequentemente positivo), classes naturais mostram NDBI fortemente negativo. Se uma caixa é muito larga ou tem whiskers extremos, verifique `stats_box.df` para o `count` daquela classe — recortes de cidades pequenas como Juiz de Fora podem ter poucos pixels em algumas classes LCZ, tornando a distribuição instável. `stats_box.fig` é a figura Plotly; chame `.show()` nela interativamente.

## 5. Estatísticas por classe LCZ: gráfico de barras com intervalos de confiança

`plot_type="bar"` renderiza as mesmas estatísticas subjacentes como uma barra de média ± intervalo
de confiança de 95% por classe — frequentemente mais fácil de ler que um boxplot quando você se
importa principalmente com tendência central e se as médias das classes são estatisticamente
distinguíveis (ICs não sobrepostos são um sinal informal aproximado de uma diferença real; o
gráfico de tamanho de efeito na próxima seção é a versão rigorosa dessa comparação).

In [6]:
stats_bar = lcz_cal_indices(lcz_map_path, idx, plot_type="bar", confidence=0.95)
stats_bar.df.select(["parameter", "lcz", "lcz_name", "mean", "ci_lower", "ci_upper"]).head(10)

parameter,lcz,lcz_name,mean,ci_lower,ci_upper
str,i16,str,f64,f64,f64
"""NDVI""",5,"""Open midrise""",0.262641,0.244103,0.281178
"""NDVI""",6,"""Open lowrise""",0.330289,0.318259,0.342318
"""NDVI""",8,"""Large lowrise""",0.221593,0.164625,0.278561
"""NDVI""",9,"""Sparsely built""",0.382647,0.354728,0.410565
"""NDVI""",11,"""Dense trees""",0.519146,0.513429,0.524863
"""NDVI""",12,"""Scattered trees""",0.538883,0.516287,0.561479
"""NDVI""",14,"""Low plants""",0.35646,0.337072,0.375848
"""NDVI""",15,"""Bare rock or paved""",0.437761,0.419667,0.455854
"""NDVI""",17,"""Water""",0.007838,-0.038476,0.054152


As barras de erro são um intervalo de confiança de distribuição t bicaudal sobre a média (`confidence=0.95` por padrão) — mais estreitas para classes com mais pixels, mais largas para as pouco amostradas. Note a simplificação sinalizada como ponytail no código-fonte: esses ICs assumem pixels independentes, sem correção de autocorrelação espacial, então trate-os como um sinal descritivo rápido em vez de um teste inferencial de nível de publicação.

## 6. Tamanhos de efeito: d de Cohen (construído vs. natural)

Esta é a versão estatisticamente rigorosa de "LCZ 2 parece menos verde que LCZ 6 no boxplot."
`plot_type="effect_size"` calcula o d de Cohen — a diferença de médias padronizada — entre todos os
pixels construídos (LCZ 1-10 agrupados) e todos os pixels naturais (LCZ 11-17 agrupados) para cada
índice solicitado, e os classifica em um gráfico de barras horizontal com linhas de referência nos
limiares convencionais (|d| = 0,2 pequeno, 0,5 médio, 0,8 grande). A tabela subjacente é retornada
separadamente em `.cohens_d`, permitindo construir comparações personalizadas classe-a-classe (por
exemplo, LCZ 2 vs. LCZ 6 especificamente) a partir dos mesmos arrays de pixels brutos, se
necessário.

In [7]:
stats_effect = lcz_cal_indices(lcz_map_path, idx, plot_type="effect_size")
stats_effect.cohens_d.sort("cohens_d", descending=True)

parameter,cohens_d,magnitude,mean_urban,mean_natural,n_urban,n_natural
str,f64,str,f64,f64,i64,i64
"""NDBI""",1.205323,"""Large""",-0.07751,-0.197633,548,1991
"""MNDWI""",0.412449,"""Small""",-0.243131,-0.269612,548,1991
"""EVI""",-1.061305,"""Large""",0.367135,0.561653,548,1991
"""NDVI""",-1.111909,"""Large""",0.324827,0.481612,548,1991


Espere que NDBI e NDVI mostrem os maiores |d| — são, quase por construção, os índices mais diagnósticos da divisão urbano/natural que o agrupamento LCZ 1-10 vs. 11-17 codifica. Um |d| grande (>0,8) para NDVI, por exemplo, permite afirmar com precisão: "classes LCZ construídas têm significativamente menos vegetação que classes naturais, com tamanho de efeito grande" — com as colunas `mean_urban`, `mean_natural`, `n_urban` e `n_natural` como o comprovante por trás dessa afirmação. Um |d| pequeno para um índice significa que ele não distingue bem urbano de natural nesta cena (o que pode acontecer com índices da família água em uma cidade continental com pouca água aberta, como Juiz de Fora).

## 7. Bônus: `lcz_cal_indexes`, uma ferramenta geral de estatística zonal

O `lcz_cal_indices` acima é feito sob medida para pilhas de índices espectrais. Mas estatística
zonal por classe LCZ — "resumir os valores de pixel deste raster por classe" — é uma operação
genérica que aparece em todo o LCZ4py: para temperatura ERA5 em grade, precipitação CHIRPS, rasters
de poluição, e mais (todos cobertos no notebook 07). `lcz_cal_indexes` é essa ferramenta geral,
construída em torno de `LCZGridResult` (o tipo de retorno compartilhado por todo downloader
`lcz_grid_*`) em vez de `LCZIndicesResult`.

Para mostrar que é um substituto direto para qualquer variável única em grade, reaproveitamos a
banda NDVI que já calculamos, envolvendo-a em um `LCZGridResult` mínimo (apenas `array`, `bands`,
`variables`) em vez de rebaixar qualquer coisa.

In [8]:
from LCZ4py._internal._lcz_grid_raster_base import LCZGridResult
from LCZ4py import lcz_cal_indexes

ndvi_i = idx.indices.index("NDVI")
ndvi_grid = LCZGridResult(array=idx.array[ndvi_i][None, ...], bands=["NDVI"], variables=["NDVI"])

ndvi_zonal = lcz_cal_indexes(lcz_map_path, ndvi_grid, variable_name="NDVI")
ndvi_zonal.df

lcz,n_pixels,mean,std,min,max,median,lcz_name,color
i16,u64,f64,f64,f64,f64,f64,str,str
5,101,0.262641,0.093903,0.079074,0.484422,0.259452,"""Open midrise""","""#FF6628"""
6,306,0.330289,0.106936,-0.043363,0.569071,0.338026,"""Open lowrise""","""#FF985E"""
8,22,0.221593,0.128487,0.080021,0.5624,0.196923,"""Large lowrise""","""#BBBBBB"""
9,119,0.382647,0.153795,-0.066413,0.635571,0.421838,"""Sparsely built""","""#FFCBAB"""
11,1260,0.519146,0.103436,0.035975,0.722769,0.530247,"""Dense trees""","""#006A18"""
12,88,0.538883,0.106647,0.216568,0.69024,0.555472,"""Scattered trees""","""#00A926"""
14,244,0.35646,0.153747,-0.070995,0.671183,0.372455,"""Low plants""","""#B5DA7F"""
15,389,0.437761,0.18151,0.065242,0.692406,0.504288,"""Bare rock or paved""","""#000000"""
17,10,0.007838,0.064743,-0.057543,0.167604,-0.014479,"""Water""","""#656BFA"""


A tabela de saída (`n_pixels, mean, std, min, max, median` por classe LCZ) tem estrutura idêntica à que você obteria resumindo temperatura ERA5 ou chuva CHIRPS no notebook 07 — mesma função, mesmo gráfico de barras por classe, apenas uma variável diferente conectada. Compare as médias por classe de `ndvi_zonal.df` com `stats_box.df` filtrado para `parameter == "NDVI"` — devem coincidir, já que ambos são médias zonais da mesma banda NDVI subjacente.

## Conclusão e próximos passos

Você baixou uma cena Sentinel-2, calculou NDVI/NDBI/MNDWI/EVI, e resumiu por classe LCZ com
estatísticas descritivas, gráficos de box/barra, e tamanhos de efeito rigorosos de d de Cohen —
transformando "classes urbanas parecem menos verdes" em uma afirmação quantificada e defensável.
Você também viu que a mesma engrenagem de estatística zonal (`lcz_cal_indexes`) generaliza para
qualquer variável em grade, não apenas índices espectrais.

**Anterior:** [`04_remote_sensing_lst_pc`](./04_remote_sensing_lst_pc.pt.ipynb) — temperatura da
superfície terrestre a partir de GOES/Sentinel-3, e fundamentos do Planetary Computer.

**Próximo:** [`06_urban_canopy_parameters`](./06_urban_canopy_parameters.pt.ipynb) — download e
processamento de parâmetros de dossel urbano (altura de edifícios, impermeabilização, fração de
vegetação) a partir de conjuntos de dados globais, complementares à visão de índices espectrais
deste notebook.